In [ ]:
import asyncio
import logging
import sqlite3
import time
import os
import random
import string
from datetime import datetime

from aiogram import Bot, Dispatcher, types, F
from aiogram.client.session.aiohttp import AiohttpSession
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.fsm.storage.memory import MemoryStorage
from aiogram.types import ReplyKeyboardMarkup, KeyboardButton, InlineKeyboardMarkup, InlineKeyboardButton, CallbackQuery

# --- НАСТРОЙКИ ---
API_TOKEN = '8534407512:AAELIASKEdgEaYWpAUOGg1XyE-Omhl7N_d4'
DB_NAME = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'flashcards.db')

INSTANCE_ID = ''.join(random.choices(string.ascii_uppercase + string.digits, k=4))
logging.basicConfig(level=logging.INFO)
notified_cards = set()

# --- СОСТОЯНИЯ (FSM) ---
class Form(StatesGroup):
    new_folder_name = State()
    choose_folder = State()
    card_question = State()
    card_answer = State()
    study_process = State()
    # Состояния для управления и удаления
    manage_choose_folder = State()
    delete_folder_choose = State()
    delete_folder_confirm = State()
    # Редактирование
    edit_new_question = State()
    edit_new_answer = State()

# --- БАЗА ДАННЫХ ---
def db_start():
    print(f"--- [{INSTANCE_ID}] Инициализация БД... ---")
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute('''CREATE TABLE IF NOT EXISTS folders (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    user_id INTEGER,
                    name TEXT)''')
    cur.execute('''CREATE TABLE IF NOT EXISTS cards (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    folder_id INTEGER,
                    question_text TEXT,
                    question_photo TEXT,
                    answer TEXT,
                    next_review REAL,
                    FOREIGN KEY(folder_id) REFERENCES folders(id))''')
    conn.commit()
    conn.close()

# --- ПЛАНИРОВЩИК ---
async def scheduler(current_bot):
    global notified_cards
    while True:
        try:
            conn = sqlite3.connect(DB_NAME)
            cur = conn.cursor()
            cur.execute('''
                SELECT c.id, f.user_id, f.name FROM cards c
                JOIN folders f ON c.folder_id = f.id
                WHERE c.next_review <= ?
            ''', (time.time(),))
            all_due_cards = cur.fetchall()
            conn.close()
            current_due_ids = {row[0] for row in all_due_cards}
            new_due_cards = [row for row in all_due_cards if row[0] not in notified_cards]
            notified_cards = current_due_ids
            for (c_id, user_id, folder_name) in new_due_cards:
                try:
                    await current_bot.send_message(user_id, f"⏰ Пора повторить карточки в папке «{folder_name}»!")
                except: pass
        except Exception as e:
            print(f"⚠ Ошибка планировщика: {e}")
        await asyncio.sleep(60)

# --- МЕНЮ ---
def main_menu():
    kb = [
        [KeyboardButton(text="📚 Учить"), KeyboardButton(text="➕ Новая карточка")],
        [KeyboardButton(text="🗂 Мои карточки"), KeyboardButton(text="📊 Моя статистика")],
        [KeyboardButton(text="📂 Создать папку"), KeyboardButton(text="🗑 Удалить папку")]
    ]
    return ReplyKeyboardMarkup(keyboard=kb, resize_keyboard=True)

# --- ФУНКЦИЯ ДЛЯ ПРОСМОТРА КАРТОЧЕК ---
async def send_card_view(bot_obj, chat_id, folder_id, index):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT id FROM cards WHERE folder_id = ?", (folder_id,))
    cards_ids = cur.fetchall()

    if not cards_ids:
        await bot_obj.send_message(chat_id, "В этой папке нет карточек.", reply_markup=main_menu())
        conn.close()
        return

    total = len(cards_ids)
    safe_index = index % total
    c_id = cards_ids[safe_index][0]
    cur.execute("SELECT question_text, answer FROM cards WHERE id = ?", (c_id,))
    q_text, answer = cur.fetchone()
    conn.close()

    text = f"<b>Карточка {safe_index + 1}/{total}</b>\n\n❓ <b>В:</b> {q_text}\n❗ <b>О:</b> {answer}"

    # Создаем клавиатуру с кнопкой закрытия во втором ряду
    kb = InlineKeyboardMarkup(inline_keyboard=[
        [
            InlineKeyboardButton(text="⬅️", callback_data=f"man:prev:{folder_id}:{safe_index}"),
            InlineKeyboardButton(text="🗑", callback_data=f"man:del:{folder_id}:{safe_index}:{c_id}"),
            InlineKeyboardButton(text="➡️", callback_data=f"man:next:{folder_id}:{safe_index}")
        ],
        [
            InlineKeyboardButton(text="✅ Завершить просмотр", callback_data="man:close")
        ]
    ])

    await bot_obj.send_message(chat_id, text, reply_markup=kb, parse_mode="HTML")

# --- ХЕНДЛЕРЫ ---
dp = Dispatcher(storage=MemoryStorage())

@dp.message(Command("start"))
async def cmd_start(message: types.Message):
    await message.answer(f"Бот запущен (ID: {INSTANCE_ID})!", reply_markup=main_menu())

# --- ПРОСМОТР И УДАЛЕНИЕ КАРТОЧЕК ---
@dp.message(F.text == "🗂 Мои карточки")
async def manage_cards_start(message: types.Message, state: FSMContext):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT name FROM folders WHERE user_id = ?", (message.from_user.id,))
    folders = cur.fetchall()
    conn.close()
    if not folders:
        await message.answer("У вас нет папок.")
        return
    kb = ReplyKeyboardMarkup(keyboard=[[KeyboardButton(text=f"📁 {f[0]}")] for f in folders], resize_keyboard=True)
    await message.answer("Выберите папку для просмотра:", reply_markup=kb)
    await state.set_state(Form.manage_choose_folder)

@dp.message(Form.manage_choose_folder)
async def manage_folder_selected(message: types.Message, state: FSMContext, bot: Bot):
    f_name = message.text.replace("📁 ", "")
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT id FROM folders WHERE user_id = ? AND name = ?", (message.from_user.id, f_name))
    res = cur.fetchone()
    conn.close()
    if res:
        await state.clear()
        await send_card_view(bot, message.chat.id, res[0], 0)

@dp.callback_query(F.data.startswith("man:"))
async def handle_callback(callback: CallbackQuery, bot: Bot):
    parts = callback.data.split(":")
    action, f_id, idx = parts[1], int(parts[2]), int(parts[3])
    await callback.message.delete()
    if action == "next": await send_card_view(bot, callback.message.chat.id, f_id, idx + 1)
    elif action == "prev": await send_card_view(bot, callback.message.chat.id, f_id, idx - 1)
    elif action == "del":
        c_id = int(parts[4])
        conn = sqlite3.connect(DB_NAME)
        cur = conn.cursor()
        cur.execute("DELETE FROM cards WHERE id = ?", (c_id,))
        conn.commit()
        conn.close()
        await callback.answer("Карточка удалена")
        await send_card_view(bot, callback.message.chat.id, f_id, idx)

# --- УДАЛЕНИЕ ПАПКИ ---
@dp.message(F.text == "🗑 Удалить папку")
async def delete_folder_start(message: types.Message, state: FSMContext):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT name FROM folders WHERE user_id = ?", (message.from_user.id,))
    folders = cur.fetchall()
    conn.close()
    if not folders:
        await message.answer("Папок нет.")
        return
    kb = ReplyKeyboardMarkup(keyboard=[[KeyboardButton(text=f"❌ {f[0]}")] for f in folders], resize_keyboard=True)
    await message.answer("Какую папку УДАЛИТЬ?", reply_markup=kb)
    await state.set_state(Form.delete_folder_choose)

@dp.message(Form.delete_folder_choose)
async def delete_folder_confirm(message: types.Message, state: FSMContext):
    f_name = message.text.replace("❌ ", "")
    await state.update_data(del_f_name=f_name)
    kb = ReplyKeyboardMarkup(keyboard=[[KeyboardButton(text="ДА, УДАЛИТЬ"), KeyboardButton(text="ОТМЕНА")]], resize_keyboard=True)
    await message.answer(f"⚠ Вы уверены, что хотите удалить папку «{f_name}»?", reply_markup=kb)
    await state.set_state(Form.delete_folder_confirm)

@dp.message(Form.delete_folder_confirm)
async def delete_folder_final(message: types.Message, state: FSMContext):
    if message.text == "ДА, УДАЛИТЬ":
        data = await state.get_data()
        conn = sqlite3.connect(DB_NAME)
        cur = conn.cursor()
        cur.execute("SELECT id FROM folders WHERE user_id = ? AND name = ?", (message.from_user.id, data['del_f_name']))
        f_id = cur.fetchone()[0]
        cur.execute("DELETE FROM cards WHERE folder_id = ?", (f_id,))
        cur.execute("DELETE FROM folders WHERE id = ?", (f_id,))
        conn.commit()
        conn.close()
        await message.answer("Папка удалена.", reply_markup=main_menu())
    else:
        await message.answer("Отменено.", reply_markup=main_menu())
    await state.clear()

# --- СТАТИСТИКА, СОЗДАНИЕ ПАПОК И КАРТОЧЕК ---
@dp.message(F.text == "📊 Моя статистика")
async def show_stats(message: types.Message):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM folders WHERE user_id = ?", (message.from_user.id,))
    f_c = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM cards c JOIN folders f ON c.folder_id = f.id WHERE f.user_id = ?", (message.from_user.id,))
    c_c = cur.fetchone()[0]
    conn.close()
    await message.answer(f"📊 Статистика:\n📂 Папок: {f_c}\n🃏 Карточек: {c_c}")

@dp.message(F.text == "📂 Создать папку")
async def create_folder(message: types.Message, state: FSMContext):
    await message.answer("Введите название:")
    await state.set_state(Form.new_folder_name)

@dp.message(Form.new_folder_name)
async def create_folder_save(message: types.Message, state: FSMContext):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("INSERT INTO folders (user_id, name) VALUES (?,?)", (message.from_user.id, message.text))
    conn.commit()
    conn.close()
    await message.answer(f"Папка «{message.text}» создана!", reply_markup=main_menu())
    await state.clear()

@dp.message(F.text == "➕ Новая карточка")
async def create_card_start(message: types.Message, state: FSMContext):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT name FROM folders WHERE user_id = ?", (message.from_user.id,))
    folders = cur.fetchall()
    conn.close()
    if not folders:
        await message.answer("Создайте папку!")
        return
    kb = ReplyKeyboardMarkup(keyboard=[[KeyboardButton(text=f"📂 {f[0]}")] for f in folders], resize_keyboard=True)
    await message.answer("Выберите папку:", reply_markup=kb)
    await state.set_state(Form.choose_folder)

@dp.message(Form.choose_folder)
async def card_f_selected(message: types.Message, state: FSMContext):
    f_name = message.text.replace("📂 ", "")
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("SELECT id FROM folders WHERE user_id = ? AND name = ?", (message.from_user.id, f_name))
    res = cur.fetchone()
    conn.close()
    if res:
        await state.update_data(folder_id=res[0])
        await message.answer("Напишите ВОПРОС:", reply_markup=types.ReplyKeyboardRemove())
        await state.set_state(Form.card_question)

@dp.message(Form.card_question)
async def card_q_step(message: types.Message, state: FSMContext):
    await state.update_data(q_text=message.text)
    await message.answer("Напишите ОТВЕТ:")
    await state.set_state(Form.card_answer)

@dp.message(Form.card_answer)
async def card_a_step(message: types.Message, state: FSMContext):
    data = await state.get_data()
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("INSERT INTO cards (folder_id, question_text, answer, next_review) VALUES (?,?,?,?)",
                (data['folder_id'], data['q_text'], message.text, time.time()))
    conn.commit()
    conn.close()
    await message.answer("Карточка готова!", reply_markup=main_menu())
    await state.clear()

# --- ОБУЧЕНИЕ ---
@dp.message(F.text == "📚 Учить")
async def start_study(message: types.Message, state: FSMContext):
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute('''SELECT c.id, c.question_text, c.answer FROM cards c
                   JOIN folders f ON c.folder_id = f.id
                   WHERE f.user_id = ? AND c.next_review <= ? LIMIT 1''', (message.from_user.id, time.time()))
    card = cur.fetchone()
    conn.close()
    if not card:
        await message.answer("Всё выучено!", reply_markup=main_menu())
        return
    await state.update_data(current_card_id=card[0], current_answer=card[2])
    kb = ReplyKeyboardMarkup(keyboard=[[KeyboardButton(text="👁 Показать ответ")]], resize_keyboard=True)
    await message.answer(f"❓ Вопрос:\n\n{card[1]}", reply_markup=kb)
    await state.set_state(Form.study_process)

@dp.message(Form.study_process, F.text == "👁 Показать ответ")
async def show_ans(message: types.Message, state: FSMContext):
    data = await state.get_data()
    kb = ReplyKeyboardMarkup(keyboard=[[KeyboardButton(text="✅ Знаю (1 день)"), KeyboardButton(text="❌ Забыл (10 мин)")]], resize_keyboard=True)
    await message.answer(f"❗ Ответ:\n\n{data['current_answer']}", reply_markup=kb)

@dp.message(Form.study_process)
async def study_res(message: types.Message, state: FSMContext):
    if message.text not in ["✅ Знаю (1 день)", "❌ Забыл (10 мин)"]: return
    data = await state.get_data()
    next_t = time.time() + 86400 if "Знаю" in message.text else time.time() + 600
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("UPDATE cards SET next_review = ? WHERE id = ?", (next_t, data['current_card_id']))
    conn.commit()
    conn.close()
    await start_study(message, state)

# --- ЗАПУСК ---
async def start_bot():
    db_start()
    proxy_url = "http://proxy.server:3128"
    session = AiohttpSession(proxy=proxy_url)
    current_bot = Bot(token=API_TOKEN, session=session)
    await current_bot.delete_webhook(drop_pending_updates=True)
    asyncio.create_task(scheduler(current_bot))
    print(f"🚀 БОТ ЗАПУЩЕН (ID: {INSTANCE_ID})")
    try: await dp.start_polling(current_bot)
    finally: await current_bot.session.close()

if __name__ == '__main__':
    try: asyncio.run(start_bot())
    except (KeyboardInterrupt, SystemExit): print("Остановлен.")